In [8]:
import pandas as pd
import numpy as np
import re
import pickle
import random
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Downloads necessários para o NLTK (corre apenas uma vez)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Seed para garantir reprodutibilidade
np.random.seed(42)
random.seed(42)

# Mapeamento das classes (0 a 4)
CLASS_MAP = {
    'Human': 0,
    'OpenAI': 1,
    'Google': 2,
    'Meta': 3,
    'Anthropic': 4
}

[nltk_data] Downloading package punkt to C:\Users\Gonçalo
[nltk_data]     Corais\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Gonçalo
[nltk_data]     Corais\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Gonçalo
[nltk_data]     Corais\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [9]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_and_slice(text, min_words=80, max_words=120):
    if not isinstance(text, str):
        return None
        
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    
    # Se o texto for menor que 80 palavras, ignorar
    if len(words) < min_words:
        return None
        
    slice_length = random.randint(min_words, max_words)
    
    if len(words) <= slice_length:
        fragment_words = words
    else:
        max_start_idx = len(words) - slice_length
        start_idx = random.randint(0, max_start_idx)
        fragment_words = words[start_idx : start_idx + slice_length]
        
    # Lematização
    processed = [lemmatizer.lemmatize(w.lower()) for w in fragment_words if w.isalnum()]
    
    return ' '.join(processed) if len(processed) > 3 else None

In [10]:
data_list = []

# Mistral (Mistral7B_CME_v1 a v4) - Classe 4
mistral_files = ['datasets/Mistral7B_CME_v1.csv', 'datasets/Mistral7B_CME_v2.csv', 'datasets/Mistral7B_CME_v3.csv', 'datasets/Mistral7B_CME_v4.csv']
for f in mistral_files:
    df_m = pd.read_csv(f)
    # Apenas textos gerados por IA (generated == 1)
    df_m = df_m[df_m['generated'] == 1]
    for text in df_m['text'].dropna():
        frag = clean_and_slice(text)
        if frag: data_list.append({'text': frag, 'label': CLASS_MAP['Mistral']})

# Google (gemini_3, gemma_1, gemma_2) - Classe 2
google_files = ['datasets/gemma_1.csv', 'datasets/gemma_2.csv', 'datasets/gemini_3.csv']
for f in google_files:
    df_g = pd.read_csv(f)
    for text in df_g['rewritten_text'].dropna():
        frag = clean_and_slice(text)
        if frag: data_list.append({'text': frag, 'label': CLASS_MAP['Google']})

#OpenAI & Human (GPT.csv) - Classes 1 e 0
df_gpt = pd.read_csv('datasets/GPT.csv')
for _, row in df_gpt.iterrows():
    frag = clean_and_slice(row['abstract'])
    if frag:
        label = CLASS_MAP['OpenAI'] if row['is_ai_generated'] == 1 else CLASS_MAP['Human']
        data_list.append({'text': frag, 'label': label})

#Meta / Llama (llama.csv) - Classe 3
df_llama = pd.read_csv('datasets/llama.csv')
for text in df_llama['answer_content'].dropna():
    
    # APLICAÇÃO DO MULTI-SLICING: Tirar 15 fragmentos de cada resposta do Llama
    # Como a função `clean_and_slice` usa random, teremos pedaços diferentes da mesma resposta longa
    for _ in range(15):
        frag = clean_and_slice(text)
        if frag: 
            data_list.append({'text': frag, 'label': CLASS_MAP['Meta']})

# Criar o DataFrame final unificado
df_all = pd.DataFrame(data_list)
# O drop_duplicates garante que, se por acaso a função extrair duas vezes o mesmo pedaço, ele é ignorado
df_all = df_all.drop_duplicates(subset=['text']) 

print("Distribuição original após Multi-Slicing no Llama:\n", df_all['label'].value_counts())

KeyError: 'Mistral'

In [ ]:
# Encontrar a classe com o menor número de exemplos
min_samples = df_all['label'].value_counts().min()

# Se min_samples for muito grande (ex: > 5000), podemos limitar para treinar mais rápido
# Como tens de entregar o projeto rapidamente, 1000 a 2000 amostras por classe é um bom alvo.
SAMPLES_PER_CLASS = min(min_samples, 2000) 

# Balancear: retirar SAMPLES_PER_CLASS de cada label
df_balanced = df_all.groupby('label').sample(n=SAMPLES_PER_CLASS, random_state=42).reset_index(drop=True)

print(f"Distribuição após balanceamento ({SAMPLES_PER_CLASS} por classe):\n", df_balanced['label'].value_counts())

Distribuição após balanceamento (1953 por classe):
 label
0    1953
1    1953
2    1953
3    1953
4    1953
Name: count, dtype: int64


In [ ]:
# Criar o vetorizador TF-IDF (limita as features para não esgotar a RAM com a tua rede Numpy)
vectorizer = TfidfVectorizer(max_features=1500)

# Transformar os textos na matriz tabular (features)
X_tabular = vectorizer.fit_transform(df_balanced['text']).toarray()
with open('Subm1/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
y = df_balanced['label'].values

# Dividir em Treino (80%), Validação (10%) e Teste (10%)
X_train, X_temp, y_train, y_temp = train_test_split(X_tabular, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Formato dos dados de Treino (Numpy Array Tabular):", X_train.shape)
print("Formato das Labels de Treino:", y_train.shape)

# Guardar os dados processados para usar no script Numpy (Tarefa 2)
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

print("Dataset tabular criado e guardado com sucesso!")

Formato dos dados de Treino (Numpy Array Tabular): (7812, 1500)
Formato das Labels de Treino: (7812,)
Dataset tabular criado e guardado com sucesso!
